# Orders EDA

This notebook contains the exploratory data analysis portion of the project using:

* `databricks_cat.silver.orders_silver` as the primary dataset
* `databricks_cat.gold.factorders` as optional downstream context

It focuses on:
* table availability and loading
* row counts and schema review
* null and distinct analysis
* numeric distributions and descriptive statistics
* quartiles and IQR-based outlier detection
* monthly trends and top entities
* concise EDA findings for reporting

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row

SILVER_TABLE = "databricks_cat.silver.orders_silver"
GOLD_TABLE = "databricks_cat.gold.factorders"

print("EDA notebook configured for:")
print(f" - {SILVER_TABLE}")
print(f" - {GOLD_TABLE}")

In [0]:
silver_exists = spark.catalog.tableExists(SILVER_TABLE)
gold_exists = spark.catalog.tableExists(GOLD_TABLE)

if not silver_exists:
    raise ValueError(f"Required table not found: {SILVER_TABLE}")

orders_df = spark.table(SILVER_TABLE)
orders_df.createOrReplaceTempView("orders_silver_v")

factorders_df = spark.table(GOLD_TABLE) if gold_exists else None
if gold_exists:
    factorders_df.createOrReplaceTempView("factorders_v")

availability_df = spark.createDataFrame([
    Row(table_name=SILVER_TABLE, available=silver_exists, role="primary silver dataset"),
    Row(table_name=GOLD_TABLE, available=gold_exists, role="optional gold context table")
])

display(availability_df)
display(orders_df.limit(10))
if gold_exists:
    display(factorders_df.limit(10))
else:
    print("Gold fact table is not available. EDA will continue with the silver table only.")

## EDA scope

This analysis focuses on the following columns requested for the project:

* `quantity`
* `total_amount`
* `order_date`
* `customer_id`
* `product_id`

The cells below provide both analytical outputs and visuals/tables suitable for screenshots in the final report.

In [0]:
orders_row_count = orders_df.count()

orders_profile_df = orders_df.agg(
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date"),
    F.countDistinct("order_id").alias("distinct_orders"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.countDistinct("product_id").alias("distinct_products")
)

print(f"Orders row count: {orders_row_count:,}")
orders_df.printSchema()
display(orders_profile_df)

if gold_exists:
    factorders_profile_df = factorders_df.agg(
        F.count("*").alias("fact_row_count"),
        F.min("order_date").alias("min_order_date"),
        F.max("order_date").alias("max_order_date"),
        F.countDistinct("DimCustomerKey").alias("distinct_dim_customers"),
        F.countDistinct("DimProductKey").alias("distinct_dim_products")
    )
    display(factorders_profile_df)

In [0]:
null_distinct_df = orders_df.agg(
    F.count("*").alias("row_count"),
    F.sum(F.when(F.col("quantity").isNull(), 1).otherwise(0)).alias("quantity_nulls"),
    F.sum(F.when(F.col("total_amount").isNull(), 1).otherwise(0)).alias("total_amount_nulls"),
    F.sum(F.when(F.col("order_date").isNull(), 1).otherwise(0)).alias("order_date_nulls"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("customer_id_nulls"),
    F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0)).alias("product_id_nulls"),
    F.countDistinct("order_id").alias("order_id_distinct"),
    F.countDistinct("customer_id").alias("customer_id_distinct"),
    F.countDistinct("product_id").alias("product_id_distinct")
)

display(null_distinct_df)

In [0]:
numeric_summary_df = orders_df.select("quantity", "total_amount").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
)

display(numeric_summary_df)

display(
    orders_df.select("quantity", "total_amount")
    .orderBy(F.desc("total_amount"))
    .limit(20)
)

In [0]:
quartile_row = orders_df.agg(
    F.expr("percentile_approx(quantity, array(0.25, 0.50, 0.75), 10000)").alias("quantity_quartiles"),
    F.expr("percentile_approx(total_amount, array(0.25, 0.50, 0.75), 10000)").alias("amount_quartiles")
).first()

quantity_q1, quantity_median, quantity_q3 = [float(x) for x in quartile_row["quantity_quartiles"]]
amount_q1, amount_median, amount_q3 = [float(x) for x in quartile_row["amount_quartiles"]]

quantity_iqr = quantity_q3 - quantity_q1
amount_iqr = amount_q3 - amount_q1

quantity_lower = quantity_q1 - (1.5 * quantity_iqr)
quantity_upper = quantity_q3 + (1.5 * quantity_iqr)
amount_lower = amount_q1 - (1.5 * amount_iqr)
amount_upper = amount_q3 + (1.5 * amount_iqr)

quartile_bounds_df = spark.createDataFrame([
    Row(metric="quantity", q1=quantity_q1, median=quantity_median, q3=quantity_q3, iqr=quantity_iqr, lower_bound=quantity_lower, upper_bound=quantity_upper),
    Row(metric="total_amount", q1=amount_q1, median=amount_median, q3=amount_q3, iqr=amount_iqr, lower_bound=amount_lower, upper_bound=amount_upper)
])

display(quartile_bounds_df)

outlier_summary_df = orders_df.agg(
    F.sum(F.when((F.col("quantity") < quantity_lower) | (F.col("quantity") > quantity_upper), 1).otherwise(0)).alias("quantity_outlier_rows"),
    F.sum(F.when((F.col("total_amount") < amount_lower) | (F.col("total_amount") > amount_upper), 1).otherwise(0)).alias("total_amount_outlier_rows")
)

display(outlier_summary_df)

display(
    orders_df.filter((F.col("total_amount") < amount_lower) | (F.col("total_amount") > amount_upper))
    .orderBy(F.desc("total_amount"))
    .limit(20)
)

In [0]:
monthly_trend_df = (
    orders_df
    .withColumn("order_month", F.date_trunc("month", F.col("order_date")))
    .groupBy("order_month")
    .agg(
        F.count("*").alias("orders"),
        F.sum("quantity").alias("units"),
        F.round(F.sum("total_amount"), 2).alias("revenue")
    )
    .orderBy("order_month")
)

display(monthly_trend_df)

In [0]:
top_products_df = (
    orders_df
    .groupBy("product_id")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("total_amount"), 2).alias("revenue")
    )
    .orderBy(F.desc("revenue"))
    .limit(15)
)

top_customers_df = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("total_amount"), 2).alias("revenue")
    )
    .orderBy(F.desc("revenue"))
    .limit(15)
)

display(top_products_df)
display(top_customers_df)

In [0]:
profile_row = orders_df.agg(
    F.count("*").alias("row_count"),
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.countDistinct("product_id").alias("distinct_products")
).first()

outlier_row = outlier_summary_df.first()

summary_lines = [
    "EDA findings:",
    f"- Row count: {profile_row['row_count']:,}",
    f"- Date range: {profile_row['min_order_date']} to {profile_row['max_order_date']}",
    f"- Distinct customers: {profile_row['distinct_customers']:,}",
    f"- Distinct products: {profile_row['distinct_products']:,}",
    f"- Quantity outlier rows: {outlier_row['quantity_outlier_rows']:,}",
    f"- Total amount outlier rows: {outlier_row['total_amount_outlier_rows']:,}",
    f"- Gold fact table available: {gold_exists}"
]

print("\n".join(summary_lines))